# 5. AIND Ephys Curation

Build an `AINDEPhysCurationScanConfig`, expand it with `GridScanGenerationTask`, and run the `AINDEPhysCurationTask` for each coordinate. The task itself clones [aind-ephys-curation](https://github.com/AllenNeuralDynamics/aind-ephys-curation) on first use, seeds its `data/` with the `postprocessed_<name>.zarr` folder produced by `AINDEPhysPostprocessingTask`, writes a `params_obi.json`, runs `code/run_capsule.py`, and copies the resulting `qc_<name>.npy` + `unit_classifier_<name>.csv` into the single config's `coordinate_output_root` (`obi-output/05_aind_ephys_curation/grid_scan/0/`).

Per recording the capsule does two things:
1. Default QC pass/fail using a pandas-query string against the analyzer's `quality_metrics` (default: `isi_violations_ratio < 0.5 and presence_ratio > 0.8 and amplitude_cutoff < 0.1`).
2. Auto-labelling each unit as `noise` / `sua` / `mua` via two HuggingFace skops models (`SpikeInterface/UnitRefine_*`).

## Imports and prerequisites

In [1]:
import subprocess
import sys
from pathlib import Path

import obi_one as obi

In [2]:
subprocess.run(
    [
        "uv", "pip", "install", "--python", sys.executable,
        "aind-data-schema", "huggingface-hub", "skops",
        # The SpikeInterface/UnitRefine HuggingFace models were trained with
        # sklearn 1.5.x. Pin to that range so the pickled SimpleImputer loads.
        "scikit-learn==1.5.2",
    ],
    check=True,
)

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv


Resolved 41 packages in 608ms
Uninstalled 3 packages in 111ms
Installed 3 packages in 21ms
 - pydantic==2.12.5
 + pydantic==2.11.10
 - pydantic-core==2.41.5
 + pydantic-core==2.33.2
 - scikit-learn==1.8.0
 + scikit-learn==1.5.2


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/python', 'aind-data-schema', 'huggingface-hub', 'skops', 'scikit-learn==1.5.2'], returncode=0)

## Build the scan config

In [3]:
postprocessing_output_path = (
    Path.cwd() / "../../../obi-output/04_aind_ephys_postprocessing/grid_scan/0"
).resolve()
assert postprocessing_output_path.exists(), (
    f"{postprocessing_output_path} not found. Run 04_aind_ephys_postprocessing.ipynb first."
)

scan_config = obi.AINDEPhysCurationScanConfig(
    initialize=obi.AINDEPhysCurationScanConfig.Initialize(
        postprocessing_output_path=postprocessing_output_path,
        n_jobs=1,
    ),
)

## Generate the grid scan and run the curation task

In [4]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/05_aind_ephys_curation/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)

[2026-04-29 18:14:46,516] INFO: Seeded 1 postprocessed analyzer(s) into /tmp/aind-ephys-curation/data


[2026-04-29 18:14:46,517] INFO: Running python -u run_capsule.py --params params_obi.json --n-jobs 1


[2026-04-29 18:14:59,454] INFO: 
CURATION
Curating recording: block0_None_recording1
Curation query: isi_violations_ratio < 0.5 and presence_ratio > 0.8 and amplitude_cutoff < 0.1
	Passing default QC: 0 / 10
Applying noise-neural classifier from SpikeInterface/UnitRefine_noise_neural_classifier
HTTP Request: GET https://huggingface.co/api/models/SpikeInterface/UnitRefine_noise_neural_classifier/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/SpikeInterface/UnitRefine_noise_neural_classifier/resolve/main/best_model.skops "HTTP/1.1 302 Found"
HTTP Request: HEAD https://huggingface.co/SpikeInterface/UnitRefine_noise_neural_classifier/resolve/main/metadata.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/SpikeInterface/UnitRefine_noise_neural_classifier/280cd07ff8310286ded5c0cfdb382e92e2a43a98/metadata.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/SpikeInterface/Uni

[PosixPath('../../../obi-output/05_aind_ephys_curation/grid_scan/0')]

## Inspect QC flags and unit classifications

In [5]:
import numpy as np
import pandas as pd

coord_dir = Path(grid_scan.single_configs[0].coordinate_output_root)
print("coordinate_output_root:", coord_dir)

qc_files = sorted(coord_dir.glob("qc_*.npy"))
csv_files = sorted(coord_dir.glob("unit_classifier_*.csv"))

for qc_path in qc_files:
    flags = np.load(qc_path)
    print(f"{qc_path.name}: {int(flags.sum())} / {len(flags)} units pass default QC")

for csv_path in csv_files:
    df = pd.read_csv(csv_path)
    print()
    print(csv_path.name)
    print(df)

coordinate_output_root: ../../../obi-output/05_aind_ephys_curation/grid_scan/0
qc_block0_None_recording1.npy: 0 / 10 units pass default QC

unit_classifier_block0_None_recording1.csv
  decoder_label  decoder_probability
0           sua             0.513499
1           sua             0.639853
2           mua             0.621799
3           mua             0.518425
4           mua             0.619274
5           sua             0.505997
6           mua             0.548739
7           mua             0.502879
8           sua             0.643502
9           mua             0.572040
